## Part I. Data Preparation: Transactions Table

### Data Loading and initial inspection
Load paysim dataset and inspect data stucture for further data transformation

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime, timedelta
import uuid

# Initial Set-up
np.random.seed(42)
random.seed(42)

# Read csv file from google cloud storage
file_path = "gs://paysim_dataset/raw_dataset/paysim_dataset.csv"

paysim_df = pd.read_csv(file_path)
paysim_df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [2]:
paysim_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


In [3]:
paysim_df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


### Core Column Standardization
Rename selected PaySim columns to reflect how transactions are structured in real payment platforms.

In [4]:
# Rename columns
paysim_df = paysim_df.rename(columns={
    "type": "payment_method",
    "nameOrig": "customer_id",
    "nameDest": "merchant_id",
    "isFraud": "is_fraud"

})

### Timestamp and Transaction ID Creation

Convert the `step` column into a simulated timestamp and generate a unique transaction ID for each record.

In [5]:
# Simulated timestamp
base_timestamp = datetime(2025, 1, 1)

paysim_df["timestamp"] = (
    base_timestamp
    + pd.to_timedelta(paysim_df["step"], unit="h")
    + pd.to_timedelta(np.random.randint(0, 60, len(paysim_df)), unit="m")
    + pd.to_timedelta(np.random.randint(0, 60, len(paysim_df)), unit="s")
    + pd.to_timedelta(np.random.randint(0, 1000, len(paysim_df)), unit="ms")
)
# Unique transaction id
paysim_df["transaction_id"] = [str(uuid.uuid4()) for _ in range(len(paysim_df))]

# Move transaction_id to the first column
cols = ["transaction_id"] + [col for col in paysim_df.columns if col != "transaction_id"]
paysim_df = paysim_df[cols]

paysim_df[["transaction_id", "step", "timestamp"]].head()

,transaction_id,step,timestamp
0,9ca2ee69-5f16-447d-a510-fbc8ec7ae26c,1,2025-01-01 01:38:40.304
1,417b744a-e7d0-4135-a4b5-ecc100236686,1,2025-01-01 01:51:26.588
2,29697ace-4b2f-46ab-ab1b-36d085883989,1,2025-01-01 01:28:33.607
3,c874e113-ddf7-4275-b89c-e5b378acae93,1,2025-01-01 01:14:27.397
4,9fc1e3d0-b57d-4899-827f-c17d74a10494,1,2025-01-01 01:42:43.804


## ID Standardization
Pad customer and merchant IDs to make identifier lengths consistent across records.

In [6]:
max_len_custid = paysim_df["customer_id"].str.len().max()
max_len_merchid = paysim_df["merchant_id"].str.len().max()

paysim_df["customer_id"] = paysim_df["customer_id"].str.pad(
    width=max_len_custid,
    side="right",
    fillchar="X"
)

paysim_df["merchant_id"] = paysim_df["merchant_id"].str.pad(
    width=max_len_merchid,
    side="right",
    fillchar="X"
)

In [7]:
paysim_df.head()

,transaction_id,step,payment_method,amount,customer_id,oldbalanceOrg,newbalanceOrig,merchant_id,oldbalanceDest,newbalanceDest,is_fraud,isFlaggedFraud,timestamp
0,9ca2ee69-5f16-447d-a510-fbc8ec7ae26c,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,2025-01-01 01:38:40.304
1,417b744a-e7d0-4135-a4b5-ecc100236686,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,2025-01-01 01:51:26.588
2,29697ace-4b2f-46ab-ab1b-36d085883989,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065X,0.0,0.0,1,0,2025-01-01 01:28:33.607
3,c874e113-ddf7-4275-b89c-e5b378acae93,1,CASH_OUT,181.00,C840083671X,181.0,0.00,C38997010XX,21182.0,0.0,1,0,2025-01-01 01:14:27.397
4,9fc1e3d0-b57d-4899-827f-c17d74a10494,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,2025-01-01 01:42:43.804


## Part II. Data Preparation: Dimensions Table

### Customer Dimension Table Creation
Create a customer table with additional synthetic attributes.

In [8]:
customers = paysim_df["customer_id"].unique()

customer_start_date = datetime(2023, 1, 1)
customer_end_date = datetime(2025, 1, 1)
customer_date_range_days = (customer_end_date - customer_start_date).days

customer_df = pd.DataFrame({
    "customer_id": customers,
    "home_country": np.random.choice(
        ["PH", "SG", "MY", "ID"],
        size=len(customers),
        p=[0.40, 0.25, 0.20, 0.15]
    ),
    "primary_device": np.random.choice(
        ["Mobile", "Desktop", "Tablet"],
        size=len(customers),
        p=[0.70, 0.20, 0.10]
    ),
    "customer_signup_date": [
        customer_start_date + timedelta(days=np.random.randint(0, customer_date_range_days))
        for _ in range(len(customers))
    ]
})

In [9]:
customer_df.head()

,customer_id,home_country,primary_device,customer_signup_date
0,C1231006815,SG,Desktop,2023-08-07
1,C1666544295,SG,Desktop,2024-04-25
2,C1305486145,SG,Mobile,2024-07-28
3,C840083671X,MY,Tablet,2024-06-30
4,C2048537720,PH,Mobile,2024-05-19


### Merchant Dimension Table Creation
Create a merchant table with additional synthetic attributes.

In [10]:
merchants = paysim_df["merchant_id"].unique()

merchant_start_date = datetime(2022, 12, 1)
merchant_end_date = datetime(2024, 1, 1)
merchant_date_range_days = (merchant_end_date - merchant_start_date).days

merchant_df = pd.DataFrame({
    "merchant_id": merchants,
    "merchant_country": np.random.choice(
        ["PH", "SG", "MY", "ID"],
        size=len(merchants),
        p=[0.40, 0.25, 0.20, 0.15]
    ),
    "merchant_category": np.random.choice(
        ["E-commerce", "Retail", "Travel", "SaaS", "Food Delivery"],
        size=len(merchants),
        p=[0.35, 0.25, 0.10, 0.15, 0.15]
    ),
    "merchant_onboarding_date": [
        merchant_start_date + timedelta(days=np.random.randint(0, merchant_date_range_days))
        for _ in range(len(merchants))
    ]
})

In [11]:
merchant_df.head()

,merchant_id,merchant_country,merchant_category,merchant_onboarding_date
0,M1979787155,ID,E-commerce,2023-11-09
1,M2044282225,SG,E-commerce,2023-01-11
2,C553264065X,PH,SaaS,2022-12-10
3,C38997010XX,PH,E-commerce,2023-11-11
4,M1230701703,MY,E-commerce,2023-10-06


## Part III. Feature Engineering

Additional transaction attributes are created to enrich the original PaySim dataset.

The engineered features include:
- `gateway`
- `transaction_status`
- `processing_time_seconds`
- `chargeback_flag`
- `refund_flag`

### Simulate gateway and transaction status
Each transaction is probabilistically assigned a payment gateway and transaction status to better reflect real fintech data.

In [12]:
gateway_options = ["Gateway A", "Gateway B", "Gateway C"]
gateway_probabilities = [0.40, 0.35, 0.25]

status_options = ["Completed", "Failed", "Pending"]
status_probabilities = [0.95, 0.03, 0.02]

paysim_df["gateway"] = np.random.choice(
    gateway_options,
    size=len(paysim_df),
    p=gateway_probabilities
)

paysim_df["transaction_status"] = np.random.choice(
    status_options,
    size=len(paysim_df),
    p=status_probabilities
)

### Estimate processing time
Processing time is simulated based on payment method.  
Different payment methods are assumed to have different processing speed ranges.

In [13]:
processing_time_conditions = [
    paysim_df["payment_method"] == "PAYMENT",
    paysim_df["payment_method"] == "TRANSFER",
    paysim_df["payment_method"] == "CASH_OUT",
    paysim_df["payment_method"] == "CASH_IN",
    paysim_df["payment_method"] == "DEBIT"
]

processing_time_choices = [
        np.random.uniform(2, 6, len(paysim_df)),
        np.random.uniform(10, 60, len(paysim_df)),
        np.random.uniform(5, 20, len(paysim_df)),
        np.random.uniform(1, 3, len(paysim_df)),
        np.random.uniform(3, 7, len(paysim_df))
    ]

paysim_df["processing_time_seconds"] = np.select(
    processing_time_conditions,
    processing_time_choices
).round(2)

In [14]:
paysim_df.head(100)

,transaction_id,step,payment_method,amount,customer_id,oldbalanceOrg,newbalanceOrig,merchant_id,oldbalanceDest,newbalanceDest,is_fraud,isFlaggedFraud,timestamp,gateway,transaction_status,processing_time_seconds
0,9ca2ee69-5f16-447d-a510-fbc8ec7ae26c,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.00,0.00,0,0,2025-01-01 01:38:40.304,Gateway B,Completed,4.82
1,417b744a-e7d0-4135-a4b5-ecc100236686,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.00,0.00,0,0,2025-01-01 01:51:26.588,Gateway C,Completed,2.56
2,29697ace-4b2f-46ab-ab1b-36d085883989,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065X,0.00,0.00,1,0,2025-01-01 01:28:33.607,Gateway A,Completed,25.64
3,c874e113-ddf7-4275-b89c-e5b378acae93,1,CASH_OUT,181.00,C840083671X,181.0,0.00,C38997010XX,21182.00,0.00,1,0,2025-01-01 01:14:27.397,Gateway C,Completed,7.12
4,9fc1e3d0-b57d-4899-827f-c17d74a10494,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.00,0.00,0,0,2025-01-01 01:42:43.804,Gateway A,Completed,2.23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,d3eb4fa9-2a48-4320-b848-6a60c560e448,1,TRANSFER,710544.77,C835773569X,0.0,0.00,C1359044626,738531.50,16518.36,0,0,2025-01-01 01:13:34.960,Gateway C,Completed,50.24
96,5e14779e-3c7f-4032-9cfa-4fd281032cea,1,TRANSFER,581294.26,C843299092X,0.0,0.00,C1590550415,5195482.15,19169204.93,0,0,2025-01-01 01:30:52.712,Gateway C,Completed,58.08
97,1aa59231-2a93-4f3d-bc39-7d705b697b50,1,TRANSFER,11996.58,C605982374X,0.0,0.00,C1225616405,40255.00,0.00,0,0,2025-01-01 01:47:33.298,Gateway A,Completed,53.05
98,705d665e-1a8d-49d0-be55-0d4e5df468af,1,PAYMENT,2875.10,C1412322831,15443.0,12567.90,M1651262695,0.00,0.00,0,0,2025-01-01 01:14:34.592,Gateway C,Completed,5.99


### Generate chargeback and refund indicators
Two exception flags—chargeback_flag for completed card transactions and refund_flag for other completed transactions not charged back—are probabilistically generated to simulate real-world outcomes.

In [16]:
# Initialize flags
paysim_df["chargeback_flag"] = 0
paysim_df["refund_flag"] = 0

# Chargeback logic:
# Only completed credit/debit card transactions are eligible
chargeback_condition = (
    (paysim_df["transaction_status"] == "Completed") &
    (paysim_df["payment_method"].isin(["Credit Card", "Debit Card"]))
)

rand_chargeback = np.random.rand(len(paysim_df))
paysim_df.loc[
    chargeback_condition & (rand_chargeback < 0.01),
    "chargeback_flag"
] = 1

# Refund logic:
# Only completed transactions without chargeback are eligible
refund_condition = (
    (paysim_df["transaction_status"] == "Completed") &
    (paysim_df["chargeback_flag"] == 0)
)

rand_refund = np.random.rand(len(paysim_df))
paysim_df.loc[
    refund_condition & (rand_refund < 0.03),
    "refund_flag"
] = 1

### Retrieve columns from another table
Country and device type are retrieved from the customers table and merged into paysim_df to simulate the transaction's origin country and device used.

In [17]:
paysim_df = paysim_df.merge(customer_df, on="customer_id", how="left")

### Rename Columns and Validate results

In [18]:
paysim_df = paysim_df.rename(columns={
    "home_country": "country",
    "primary_device": "device_type"

})

In [19]:
paysim_df = paysim_df[
    [
        "transaction_id",
        "timestamp",
        "customer_id",
        "merchant_id",
        "payment_method",
        "gateway",
        "amount",
        "transaction_status",
        "processing_time_seconds",
        "country",
        "device_type",
        "is_fraud",
        "chargeback_flag",
        "refund_flag"
    ]
]

## Part IV. Save Results as CSV

#### Save transactions as csv

In [21]:
paysim_df.head()

,transaction_id,timestamp,customer_id,merchant_id,payment_method,gateway,amount,transaction_status,processing_time_seconds,country,device_type,is_fraud,chargeback_flag,refund_flag
0,9ca2ee69-5f16-447d-a510-fbc8ec7ae26c,2025-01-01 01:38:40.304,C1231006815,M1979787155,PAYMENT,Gateway B,9839.64,Completed,4.82,SG,Desktop,0,0,0
1,417b744a-e7d0-4135-a4b5-ecc100236686,2025-01-01 01:51:26.588,C1666544295,M2044282225,PAYMENT,Gateway C,1864.28,Completed,2.56,SG,Desktop,0,0,0
2,29697ace-4b2f-46ab-ab1b-36d085883989,2025-01-01 01:28:33.607,C1305486145,C553264065X,TRANSFER,Gateway A,181.00,Completed,25.64,SG,Mobile,1,0,0
3,c874e113-ddf7-4275-b89c-e5b378acae93,2025-01-01 01:14:27.397,C840083671X,C38997010XX,CASH_OUT,Gateway C,181.00,Completed,7.12,MY,Tablet,1,0,0
4,9fc1e3d0-b57d-4899-827f-c17d74a10494,2025-01-01 01:42:43.804,C2048537720,M1230701703,PAYMENT,Gateway A,11668.14,Completed,2.23,PH,Mobile,0,0,0


In [22]:
transactions_csv_path = "gs://paysim_dataset/enriched_dataset/transactions.csv"
paysim_df.to_csv(transactions_csv_path, index=False)

### Save customers as csv

In [ ]:
customer_df.head()

In [ ]:
customer_csv_path = "gs://paysim_dataset/enriched_dataset/customers.csv"
customer_df.to_csv(customer_csv_path, index=False)

### Save merchants as csv

In [ ]:
merchant_df.head()

In [ ]:
merchant_csv_path = "gs://paysim_dataset/enriched_dataset/merchants.csv"
merchant_df.to_csv(merchant_csv_path, index=False)